In [ ]:
import os
import cv2
import torch
import numpy as np
import torchvision
import torch.nn as nn
from torchvision.transforms import functional as F
from torchvision import models, transforms
from torchvision.ops import box_iou, nms
from sklearn.svm import LinearSVC

# ==========================================
# SETUP: PASCAL VOC DATASET WRAPPER
# ==========================================
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair',
    'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor'
]
CLASS_TO_IDX = {cls_name: idx + 1 for idx, cls_name in enumerate(VOC_CLASSES)}

class VOCDetectionWrapper(torchvision.datasets.VOCDetection):
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        img_tensor = F.to_tensor(img)
        
        boxes, labels = [], []
        objects = target['annotation']['object']
        if not isinstance(objects, list):
            objects = [objects]
            
        for obj in objects:
            bndbox = obj['bndbox']
            # PASCAL VOC coordinates are 1-based, subtract 1
            boxes.append([
                float(bndbox['xmin']) - 1, float(bndbox['ymin']) - 1,
                float(bndbox['xmax']) - 1, float(bndbox['ymax']) - 1
            ])
            labels.append(CLASS_TO_IDX[obj['name']])
            
        return img_tensor, torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

# This will download the dataset to ./data if it doesn't exist
print("Loading/Downloading PASCAL VOC 2012...")
dataset = VOCDetectionWrapper(root='./data', year='2012', image_set='train', download=True)


# ==========================================
# STEP 1: REGION PROPOSAL GENERATION
# ==========================================
def generate_region_proposals(image_np, max_proposals=2000):
    """Uses Selective Search to generate region proposals."""
    ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    ss.setBaseImage(image_np)
    ss.switchToSelectiveSearchFast()
    rects = ss.process()
    
    proposals = []
    for (x, y, w, h) in rects[:max_proposals]:
        proposals.append([x, y, x + w, y + h])
    return np.array(proposals)


# ==========================================
# STEP 2: FEATURE EXTRACTION (PRETRAINED CNN)
# ==========================================
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(pretrained=True)
        # Remove final classification layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.eval()

    def forward(self, x):
        with torch.no_grad():
            return self.features(x).view(x.size(0), -1) # Flatten to 2048

def extract_features(image_np, proposals, model, device):
    features = []
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    for (x1, y1, x2, y2) in proposals:
        roi = image_np[y1:y2, x1:x2]
        if roi.shape[0] == 0 or roi.shape[1] == 0:
            continue
        roi_t = transform(roi).unsqueeze(0).to(device)
        feat = model(roi_t)
        features.append(feat.cpu().numpy().flatten())
    return np.array(features)


# ==========================================
# STEP 3: TRAIN CLASSIFIER (LINEAR SVM)
# ==========================================
def create_svm_labels(proposals, gt_boxes, gt_classes, iou_thresh=0.5):
    labels = []
    for prop in proposals:
        max_iou, best_class = 0, 0
        for idx, gt in enumerate(gt_boxes):
            b1 = torch.tensor(prop, dtype=torch.float).unsqueeze(0)
            b2 = torch.tensor(gt, dtype=torch.float).unsqueeze(0)
            iou = box_iou(b1, b2).item()
            
            if iou > max_iou:
                max_iou = iou
                best_class = gt_classes[idx]
                
        if max_iou >= iou_thresh:
            labels.append(best_class)
        elif max_iou < 0.3:
            labels.append(0) # Background
        else:
            labels.append(-1) # Ignore
    return np.array(labels)

def train_svm(features, labels):
    valid_idx = labels >= 0
    svm = LinearSVC(max_iter=2000, class_weight='balanced')
    svm.fit(features[valid_idx], labels[valid_idx])
    return svm


# ==========================================
# STEP 4: BOUNDING BOX REGRESSION
# ==========================================
class BBoxRegressor(nn.Module):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 4) # tx, ty, tw, th
        )

    def forward(self, x):
        return self.fc(x)

def train_regressor(model, features, targets, device, epochs=10):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    model.train()
    
    # Move the tensors to the same device as the model (GPU/cuda:0)
    feat_t = torch.tensor(features, dtype=torch.float32).to(device)
    tgt_t = torch.tensor(targets, dtype=torch.float32).to(device)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(feat_t), tgt_t)
        loss.backward()
        optimizer.step()


# ==========================================
# STEP 5: TESTING + NMS
# ==========================================
def test_rcnn(image_np, proposals, feature_model, svm, regressor, device):
    # 1. Extract Features
    features = extract_features(image_np, proposals, feature_model, device)
    
    # 2. SVM Predictions
    predictions = svm.predict(features)
    scores = svm.decision_function(features)
    
    pos_idx = np.where(predictions > 0)[0]
    if len(pos_idx) == 0: return [], []
    
    pos_boxes = proposals[pos_idx]
    pos_features = features[pos_idx]
    pos_scores = np.max(scores[pos_idx], axis=1) if len(scores.shape) > 1 else scores[pos_idx]
    
    # 3. BBox Regression Refinement
    with torch.no_grad():
        offsets = regressor(torch.tensor(pos_features, dtype=torch.float32).to(device)).cpu().numpy()
    refined_boxes = pos_boxes + offsets 
    
    # 4. NMS
    keep_idx = nms(
        torch.tensor(refined_boxes, dtype=torch.float32), 
        torch.tensor(pos_scores, dtype=torch.float32), 
        iou_threshold=0.3
    )
    
    return refined_boxes[keep_idx.numpy()], predictions[pos_idx][keep_idx.numpy()]

# --- MAIN EXECUTION PIPELINE ---
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    extractor = FeatureExtractor().to(device)
    regressor = BBoxRegressor().to(device)
    
    # Example: Processing the first image in PASCAL VOC to train
    img_tensor, gt_boxes, gt_classes = dataset[0]
    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    print("Generating proposals...")
    proposals = generate_region_proposals(img_np)
    
    print("Extracting features...")
    features = extract_features(img_np, proposals, extractor, device)
    
    print("Training SVM...")
    labels = create_svm_labels(proposals, gt_boxes.numpy(), gt_classes.numpy())
    svm_model = train_svm(features, labels)
    
    print("Training Bounding Box Regressor...")
    # For a real pipeline, target offsets must be calculated mathematically. 
    # Using dummy targets here for the pipeline flow to prevent crashing.
    dummy_targets = np.zeros((len(features), 4)) 
    train_regressor(regressor, features, dummy_targets, device)
    
    print("Testing Pipeline...")
    final_boxes, final_classes = test_rcnn(img_np, proposals, extractor, svm_model, regressor, device)
    print(f"Detected {len(final_boxes)} objects after NMS.")

Loading/Downloading PASCAL VOC 2012...


## Faster R-CNN
